# Deep Learning 039 — Hyperparameter Tuning

The lesson searched **54 configurations** and found **90.4%** best, **88.8%** for
a sensible default and **79.2%** worst — a gain of **+1.6%** over a good guess
and a spread of **11.2%**.

TensorFlow is not installed here, so this notebook does the same job with
scikit-learn and plain Python: the *pattern* (declare ranges → search → read the
table) is identical, and the arithmetic about search strategies is exact.

| Part | What we check |
|---|---|
| A | the shape of a search space: how 54 comes about |
| B | a real grid search on a small model |
| C | random search: how many trials to land in the top 5% |
| D | why tuning is insurance, not a jackpot |

In [ ]:
import itertools, numpy as np
np.random.seed(0)

## Part A — The search space

`build_model(hp)` declares *ranges*, not values. Multiply the choices together
and you have the size of the space.

In [ ]:
space = {
    "units":        [32, 64, 128],
    "layers":       [1, 2, 3],
    "activation":   ["relu", "tanh"],
    "learning_rate": [1e-2, 1e-3, 1e-4],
}
total = 1
for name, options in space.items():
    total *= len(options)
    print(f"{name:<14} {len(options)} options")
print("\ngrid search would train", total, "models")
assert total == 54

## Part B — A real search

The same loop as `tuner.search()`, on a model small enough to fit here. Each
"trial" trains one configuration and scores it on held-out data.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X, y = make_classification(n_samples=1200, n_features=20, n_informative=8,
                           n_classes=2, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

small = {"units": [8, 32], "layers": [1, 2], "activation": ["relu", "tanh"],
         "learning_rate": [1e-2, 1e-3]}

results = []
for units, layers, act, lr in itertools.product(*small.values()):
    clf = MLPClassifier(hidden_layer_sizes=(units,) * layers, activation=act,
                        learning_rate_init=lr, max_iter=120, random_state=0)
    clf.fit(X_tr, y_tr)
    results.append(((units, layers, act, lr), clf.score(X_te, y_te)))

results.sort(key=lambda r: -r[1])
print(f"{'units':>6}{'layers':>8}{'act':>7}{'lr':>8}   accuracy")
for cfg, acc in results:
    print(f"{cfg[0]:>6}{cfg[1]:>8}{cfg[2]:>7}{cfg[3]:>8}   {acc:.3f}")

best, worst = results[0][1], results[-1][1]
print(f"\nbest {best:.3f} | worst {worst:.3f} | spread {best - worst:.3f}")

Your numbers will differ from the lesson's (different data, different library),
but the shape is the same: a wide spread, with the worst configurations combining
several poor choices.

## Part C — Random search

Grid search trains every configuration. Random search samples. How many random
trials before you land in the top 5% of a 54-point space?

In [ ]:
# P(miss top 5% in n draws) = 0.95**n, so:
import math
for target in (0.5, 0.9, 0.95, 0.99):
    n = math.ceil(math.log(1 - target) / math.log(0.95))
    print(f"to be {target:>5.0%} sure of hitting the top 5%: {n:>3} random trials")

# now check it by simulation, drawing DISTINCT configurations from the 54
rng = np.random.default_rng(0)
scores = rng.normal(size=54)
cut = np.quantile(scores, 0.95)
good = int((scores >= cut).sum())
hits = [np.any(rng.choice(scores, size=14, replace=False) >= cut) for _ in range(4000)]
print(f"\ntop-5% cut covers {good} of 54 configurations ({good/54:.1%})")
print(f"formula (independent draws): {1 - 0.95**14:.1%}")
print(f"simulated (distinct draws) : {np.mean(hits):.1%}")
assert 0.55 < np.mean(hits) < 0.67

The simulation beats the formula — about **61%** against **51%** — and the gap is
real, not noise. Two reasons: a real tuner samples *distinct* configurations, so
each trial rules one out instead of risking a repeat, and the top-5% cut on 54
points actually covers 3 of them (5.6%). The formula is the pessimistic version;
treat it as a floor.

Either way the argument for random search holds: about **14 trials** for a
top-5% configuration, against 54 to be certain. The lesson's three strategies —
random, Bayesian and Hyperband — all spend fewer trials than the grid.

## Part D — Insurance, not a jackpot

In [ ]:
best, default, worst = 90.4, 88.8, 79.2
print(f"gain over a sensible default : {best - default:+.1f} points")
print(f"loss if you choose badly     : {default - worst:-.1f} points")
print(f"full spread                  : {best - worst:.1f} points")
assert round(best - default, 1) == 1.6 and round(best - worst, 1) == 11.2

Read those two numbers together. Tuning bought **1.6 points** over a reasonable
guess, but a bad guess costs **9.6**. The value is avoiding the floor, not
reaching the ceiling — and the lesson's worst configuration got there by
combining four poor choices at once.

## What to take away

- Declare **ranges**, not values, then let the search try them.
- Grid search cost grows multiplicatively: 3 × 3 × 2 × 3 = **54** models.
- Random search finds a top-5% configuration in about **14** trials, on average.
- **Activation and learning rate** are the choices that can hurt you; width is
  forgiving.

## Exercises

1. Add `batch_size` with 3 options to Part A. How many models does grid search
   train now?
2. In Part B, keep the best configuration and vary only the learning rate. How
   much of the spread was the learning rate alone?
3. Work out how many random trials you need to be 99% sure of the top **1%**.